# Sequential Training

We train the models sequentially, across multiple days.

- **Dataset:** TOTF.PA (Euronext Paris), 25 daily LOB snapshots
- **Scaler:** Quantile (Box-Cox + z-score)
- **Models:**
    - Transformer + OC-SVM (hybrid)
    - PNN (Probabilistic Neural Network)
    - PRAE (Probabilistic Robust Autoencoder)
- **Training Strategy:** For each of the first 24 days, use the first hour of market data split into alternating 5-minute blocks of training and validation.
- **Test Day:** Day 25 (held out entirely).
- **Features:** base, tao (weighted imbalance), poutre (rapidity / event flow), hawkes (memory), ofi (order flow imbalance).

In [ ]:
import os
import sys
import glob
import logging

import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
import joblib

sys.path.insert(0, os.path.abspath(".."))

from detection.data.loaders import create_sequences, load_processed, scale_and_create_loaders
from detection.data.preprocessing import split_first_hour_blocks
from detection.data.scalers import scaler as QuantileScaler
from detection.models.hybrid import TransformerOCSVM
from detection.models.prae import calculate_heuristic_lambda, calculate_reconstruction_lambda, grid_search_lambda
from detection.trainers.factory import build_fresh_model
from detection.trainers.training import train_one_block

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("training")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info("Device: %s", DEVICE)

## Configuration

In [ ]:
# Paths
DATA_DIR = os.path.join("..", "data", "processed", "TOTF.PA-book")
RESULTS_DIR = os.path.join("..", "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

# Data parameters
TIME_COL = "xltime"
MARKET_OPEN_HOUR = 9.0   # Euronext Paris continuous session
FIRST_HOUR_MINUTES = 60
TRAIN_BLOCK_MINUTES = 5
VAL_BLOCK_MINUTES = 5

# Preprocessing
SEQ_LENGTH = 25
BATCH_SIZE = 64
TARGET_COL = "log_return"

# Training
EPOCHS = 1000
LR = 1e-3
PATIENCE = 20

# Model architectures
TRANSFORMER_CFG = dict(model_dim=128, num_heads=8, num_layers=6, representation_dim=128, dim_feedforward=512)
PNN_HIDDEN_DIM = 64
PRAE_SIGMA = 0.5

# OC-SVM (Nyström approximation + linear SGD on CUDA)
OCSVM_NU = 0.01
NYSTROEM_COMPONENTS = 300
OCSVM_SGD_LR = 0.01
OCSVM_SGD_EPOCHS = 500

# LOB columns present in processed files
LOB_COLUMNS = [
    f"{side}-{typ}-{lvl}"
    for lvl in range(1, 11)
    for side, typ in [("bid","price"),("bid","volume"),("ask","price"),("ask","volume")]
]

# File listing
FILES = sorted(glob.glob(os.path.join(DATA_DIR, "*.parquet")))
NUM_TRAIN_DAYS = len(FILES) - 3  # Last 3 days are held out for testing
logger.info("Found %d daily files.  Training on %d days, testing on day %d.", len(FILES), NUM_TRAIN_DAYS, len(FILES))

2026-03-04 19:15:58 [INFO] training: Found 25 daily files.  Training on 3 days, testing on day 25.


## Sequential Training Loop

For each of the 24 training days and for each model type:

1. Load pre-processed parquet (features already computed).
2. Split the first hour into 5-min train/val blocks.
3. Scale, sequence, and build DataLoaders.
4. Train the model on that block (continuing from the previous day's weights).
5. After all days, save the final model weights and scaler.

In [ ]:
MODEL_TYPES = ["transformer_ocsvm", "pnn", "prae"]

trained_models = {}
trained_scalers = {}
feature_name_map = {}
training_log = []

for model_type in MODEL_TYPES:
    logger.info("=" * 80)
    logger.info("MODEL: %s", model_type.upper())
    logger.info("=" * 80)

    model, detector = None, None
    scaler = QuantileScaler()
    feature_names = None
    scaler_fitted = False
    prae_lambda = None

    for day_idx in range(NUM_TRAIN_DAYS):
        filepath = FILES[day_idx]
        day_name = os.path.basename(filepath)
        logger.info("Day %d/%d: %s", day_idx + 1, NUM_TRAIN_DAYS, day_name)

        df_day, features = load_processed(filepath, TIME_COL, LOB_COLUMNS)

        if feature_names is None:
            feature_names = features.columns.tolist()

        for col in feature_names:
            if col not in features.columns:
                features[col] = 0.0
        features = features[feature_names]

        train_feat, val_feat = split_first_hour_blocks(
            df_day[TIME_COL].values, features,
            MARKET_OPEN_HOUR, FIRST_HOUR_MINUTES, TRAIN_BLOCK_MINUTES, VAL_BLOCK_MINUTES)
        if len(train_feat) < SEQ_LENGTH + 1 or len(val_feat) < SEQ_LENGTH + 1:
            logger.warning("Day %d: insufficient data in first hour, skipping.", day_idx + 1)
            continue

        fit = not scaler_fitted
        train_loader, val_loader, scaler, feature_names = scale_and_create_loaders(
            train_feat, val_feat, scaler, model_type, feature_names,
            SEQ_LENGTH, BATCH_SIZE, TARGET_COL, DEVICE, fit_scaler=fit)
        scaler_fitted = True

        if train_loader is None:
            logger.warning("Day %d: empty loaders after sequencing, skipping.", day_idx + 1)
            continue

        num_features = len(feature_names)
        if model is None:
            model, detector = build_fresh_model(
                model_type, num_features, SEQ_LENGTH, DEVICE,
                TRANSFORMER_CFG, EPOCHS, LR, PATIENCE,
                OCSVM_NU, NYSTROEM_COMPONENTS, OCSVM_SGD_LR, OCSVM_SGD_EPOCHS,
                PNN_HIDDEN_DIM, PRAE_SIGMA,
            )

            if model_type == "prae":
                heuristic_lambda = calculate_heuristic_lambda(
                    train_loader, seq_len=SEQ_LENGTH, num_features=num_features)
                logger.info("PRAE heuristic lambda = %.4f", heuristic_lambda)

                prae_lambda = grid_search_lambda(
                    train_loader, val_loader,
                    heuristic_lambda=heuristic_lambda,
                    num_train_samples=len(train_loader.dataset),
                    num_features=num_features,
                    seq_len=SEQ_LENGTH,
                    device=str(DEVICE),
                    epochs=15,
                    learning_rate=LR,
                    **TRANSFORMER_CFG)
                model.lambda_reg = prae_lambda
                logger.info("PRAE lambda_reg set to %.4f (grid search, validation MSE)", prae_lambda)

        if model_type == "prae":
            n_samples = len(train_loader.dataset)
            model.mu = torch.nn.Parameter(torch.full((n_samples,), 0.5, device=DEVICE))
            # Recompute lambda relative to current reconstruction error so that
            # the gates can actually differentiate outliers from inliers even
            # when the backbone is already well-trained from prior days.
            if prae_lambda is not None:
                rec_lambda = calculate_reconstruction_lambda(model, train_loader, device=str(DEVICE))
                model.lambda_reg = rec_lambda
                logger.info("PRAE lambda_reg recalibrated to %.4f (mean rec error)", rec_lambda)

        train_one_block(model, detector, train_loader, val_loader, model_type,
                        PATIENCE, EPOCHS, LR, DEVICE)

        training_log.append({
            "model": model_type,
            "day": day_idx + 1,
            "file": day_name,
            "train_samples": len(train_loader.dataset),
            "val_samples": len(val_loader.dataset),
        })

    if model_type == "transformer_ocsvm" and detector is not None:
        logger.info("Fitting Nystrom OC-SVM on latent representations from all training days...")
        model.eval()
        all_latent = []
        with torch.no_grad():
            for day_idx in range(NUM_TRAIN_DAYS):
                filepath = FILES[day_idx]
                df_day, feats = load_processed(filepath, TIME_COL, LOB_COLUMNS)
                for col in feature_names:
                    if col not in feats.columns:
                        feats[col] = 0.0
                feats = feats[feature_names]
                train_feat, _ = split_first_hour_blocks(
                    df_day[TIME_COL].values, feats,
                    MARKET_OPEN_HOUR, FIRST_HOUR_MINUTES, TRAIN_BLOCK_MINUTES, VAL_BLOCK_MINUTES)
                if len(train_feat) < SEQ_LENGTH + 1:
                    continue
                scaled = scaler.transform(train_feat.values.astype(np.float32)).astype(np.float32)
                seqs = create_sequences(scaled, SEQ_LENGTH)
                if len(seqs) == 0:
                    continue
                x_tensor = torch.tensor(seqs, dtype=torch.float32)
                ds = TensorDataset(x_tensor, x_tensor)
                loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)
                for batch in loader:
                    x = batch[0].to(DEVICE)
                    latent = model.get_representation(x)
                    all_latent.append(latent)
        X_latent = torch.cat(all_latent, dim=0)

        gamma = TransformerOCSVM._median_heuristic_gamma(X_latent)
        detector.ocsvm.set_gamma(gamma)
        logger.info("OC-SVM gamma set to %.6f  (median heuristic, d=%d)", gamma, X_latent.shape[1])

        detector.ocsvm.fit(X_latent)
        logger.info("Nystrom OC-SVM fitted on %d latent vectors from %d days.",
                    X_latent.shape[0], NUM_TRAIN_DAYS)

    trained_models[model_type] = (model, detector)
    trained_scalers[model_type] = scaler
    feature_name_map[model_type] = feature_names

    weights_path = os.path.join(RESULTS_DIR, f"{model_type}_weights.pth")
    torch.save(model.state_dict(), weights_path)
    logger.info("Saved %s weights to %s", model_type, weights_path)

    if model_type == "transformer_ocsvm" and detector is not None:
        detector_path = os.path.join(RESULTS_DIR, f"{model_type}_detector.pth")
        torch.save(detector.ocsvm, detector_path)
        logger.info("Saved Nystrom OC-SVM detector to %s", detector_path)

    scaler_path = os.path.join(RESULTS_DIR, f"{model_type}_scaler.pkl")
    joblib.dump(scaler, scaler_path)
    logger.info("Saved scaler to %s", scaler_path)

    feat_path = os.path.join(RESULTS_DIR, f"{model_type}_features.txt")
    with open(feat_path, "w") as f:
        f.write("\n".join(feature_names))

logger.info("All models trained.")

## Training Summary

In [6]:
log_df = pd.DataFrame(training_log)
print(f"\n{'='*60}")
print("TRAINING SUMMARY")
print(f"{'='*60}")
for mt in MODEL_TYPES:
    subset = log_df[log_df["model"] == mt]
    total = subset["train_samples"].sum() + subset["val_samples"].sum()
    print(f"\n{mt.upper()}")
    print(f"  Days trained: {len(subset)}/{NUM_TRAIN_DAYS}")
    print(f"  Total samples seen: {total:,}")
    model_obj = trained_models[mt][0]
    n_params = sum(p.numel() for p in model_obj.parameters())
    print(f"  Model parameters: {n_params:,}")

print(f"\nSaved artefacts in: {RESULTS_DIR}")
print("Files:")
for f in sorted(os.listdir(RESULTS_DIR)):
    size = os.path.getsize(os.path.join(RESULTS_DIR, f))
    print(f"  {f:40s}  {size/1024:.1f} KB")

log_df


TRAINING SUMMARY

TRANSFORMER_OCSVM
  Days trained: 3/3
  Total samples seen: 367,998
  Model parameters: 1,547,481

PNN
  Days trained: 3/3
  Total samples seen: 367,998
  Model parameters: 142,659

PRAE
  Days trained: 3/3
  Total samples seen: 367,998
  Model parameters: 1,629,438

Saved artefacts in: ..\results
Files:
  pnn_features.txt                          1.9 KB
  pnn_scaler.pkl                            706.6 KB
  pnn_weights.pth                           559.4 KB
  prae_features.txt                         1.9 KB
  prae_scaler.pkl                           706.6 KB
  prae_weights.pth                          7638.7 KB
  transformer_ocsvm_detector.pkl            149801.2 KB
  transformer_ocsvm_detector.pth            505.7 KB
  transformer_ocsvm_features.txt            1.9 KB
  transformer_ocsvm_scaler.pkl              706.6 KB
  transformer_ocsvm_weights.pth             7318.1 KB


,model,day,file,train_samples,val_samples
0,transformer_ocsvm,1,2015-01-02-TOTF.PA-book.parquet,38696,50974
1,transformer_ocsvm,2,2015-01-05-TOTF.PA-book.parquet,51141,55810
2,transformer_ocsvm,3,2015-01-06-TOTF.PA-book.parquet,81957,89420
3,pnn,1,2015-01-02-TOTF.PA-book.parquet,38696,50974
4,pnn,2,2015-01-05-TOTF.PA-book.parquet,51141,55810
5,pnn,3,2015-01-06-TOTF.PA-book.parquet,81957,89420
6,prae,1,2015-01-02-TOTF.PA-book.parquet,38696,50974
7,prae,2,2015-01-05-TOTF.PA-book.parquet,51141,55810
8,prae,3,2015-01-06-TOTF.PA-book.parquet,81957,89420
